# AnyCam — 05b · Step 7 round 2: SVCT improved

Round 1: regression fixed on all cameras, cultivar not yet (best 0.64 vs 0.76 target). Round 2 changes:
- **teacher mode**: every batch = full 204-band view (teacher) + one virtual camera (student learns from teacher, weight 0.5)
- **mixed camera sampler**: 1/3 phone-like, 1/3 drone-like, 1/3 random (the 7 real cameras stay held out)
- **longer training**: up to 80 epochs, patience 30, checkpoint chosen by validation accuracy + R2

Runs (~3-4 h total): SVCT round 2 · no-consistency round 2 · no-wavelength-layer round 2 · 4 extra camera-specific models.
Setup: **HyperLeaf2024 (trophy)**, **GPU T4 x2**, **Internet On**. Before saving to GitHub check the repository is
**ShahnawazQ007/one-model-many-camera**, file `kaggle_runs/05b_step7_round2.ipynb`.

In [ ]:
import os, subprocess, glob, json, pathlib
REPO = 'https://github.com/ShahnawazQ007/one-model-many-camera.git'
if not os.path.exists('/kaggle/working/anycam'):
    subprocess.run(['git', 'clone', '-q', '--depth', '1', REPO, '/kaggle/working/anycam'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/anycam', 'pull', '-q'], check=True)
os.chdir('/kaggle/working/anycam')
!pip -q install timm tifffile imagecodecs
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - turn on the GPU accelerator!')
hits = [os.path.dirname(f) for f in glob.glob('/kaggle/input/**/train.csv', recursive=True)
        if os.path.isdir(os.path.join(os.path.dirname(f), 'images'))]
assert hits, 'Add Input -> search hyperleaf -> the one with the trophy icon'
DATA = hits[0]; print('data:', DATA, '| images:', len(glob.glob(f'{DATA}/images/*.tiff')))

In [ ]:
# 1) SVCT round 2 (ours)
!python experiments/train_svct.py --data {DATA} --stem hyper --views 2 --cons-mode teacher --cons 0.5 --radio 0.5 --sampler mixed --select acc --epochs 80 --patience 30 --out runs/svct_r2

In [ ]:
# 2) ablation: no consistency (one camera per batch, full bands 30% of the time)
!python experiments/train_svct.py --data {DATA} --stem hyper --views 1 --p-full 0.3 --radio 0.5 --sampler mixed --select acc --epochs 80 --patience 30 --out runs/svct_r2_nocons

In [ ]:
# 3) ablation: no wavelength layer (bands interpolated to 204), same teacher + sampler
!python experiments/train_svct.py --data {DATA} --stem interp --views 2 --cons-mode teacher --cons 0.5 --radio 0.5 --sampler mixed --select acc --epochs 80 --patience 30 --out runs/svct_r2_interp

In [ ]:
# first comparison (before the extra camera-specific models)
!python scripts/plot_step7.py --baseline results/step06_camera_test/camera_eval.csv --runs "SVCT r2 (ours)=runs/svct_r2" "no consistency r2=runs/svct_r2_nocons" "no wavelength layer r2=runs/svct_r2_interp" --out runs/step7_r2_compare
!for f in runs/svct_*/camera_eval.csv; do echo "### $f"; cat $f; done

In [ ]:
# 4) camera-specific models for the 4 cameras not yet done (completes the 'one model per camera' reference)
for cam in ['dji_p4_multispectral', 'parrot_sequoia', 'generic_rgb_nir', 'micasense_no_red_edge']:
    !python experiments/train_baseline.py --data {DATA} --split official --camera {cam} --upscale 2 --aug strong --epochs 50 --patience 12 --out runs/oracle_{cam}

In [ ]:
# final comparison with all 7 camera-specific models
!python scripts/plot_step7.py --baseline results/step06_camera_test/camera_eval.csv --oracles "runs/oracle_*" --runs "SVCT r2 (ours)=runs/svct_r2" "no consistency r2=runs/svct_r2_nocons" "no wavelength layer r2=runs/svct_r2_interp" --out runs/step7_r2_compare
from IPython.display import Image, display
display(Image('runs/step7_r2_compare/step7_compare.png'))
!for f in runs/oracle_*/metrics.json; do echo "### $f"; cat $f; done

## Done
**Save Version → Quick Save** → GitHub: repository **ShahnawazQ007/one-model-many-camera**, branch main,
file `kaggle_runs/05b_step7_round2.ipynb`. Then stop the session (power button) and tell Claude "done".